In [ ]:
!pip install -q --upgrade pip
!pip install -q --upgrade datasets[audio] transformers accelerate evaluate jiwer soundfile librosa

In [ ]:
from datasets import load_dataset, Audio

raw = load_dataset("tahmaz/azerbaijani-asr-zenfira_1k", "default", split="train")

common_voice = raw.rename_column("audio_file", "audio")
common_voice = common_voice.rename_column("transcript", "sentence")
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))
common_voice = common_voice.remove_columns(["duration"])

print(common_voice[0]["audio"])
print(common_voice[0]["sentence"])

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="azerbaijani", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print(device)

In [ ]:
def transcribe(audio_dict):
    feats = processor.feature_extractor(
        audio_dict["array"],
        sampling_rate=audio_dict["sampling_rate"],
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        decoder_ids = processor.get_decoder_prompt_ids(language="azerbaijani", task="transcribe")
        out = model.generate(feats, forced_decoder_ids=decoder_ids)

    return processor.tokenizer.batch_decode(out, skip_special_tokens=True)[0].strip()


sample = common_voice[0]
print(sample["sentence"])
print(transcribe(sample["audio"]))

In [ ]:
import string
import numpy as np
import pandas as pd
from jiwer import wer, cer

def normalize(text):
    text = text.lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation + '«»—–'))
    return " ".join(text.split())


MAX_SAMPLES = 20
n = min(MAX_SAMPLES, len(common_voice))
results = []

for i in range(n):
    row = common_voice[i]
    ref = row["sentence"].strip()
    if not ref:
        continue

    try:
        hyp = transcribe(row["audio"])
    except Exception as e:
        print(i, e)
        continue

    ref_n = normalize(ref)
    hyp_n = normalize(hyp) or "<bos>"

    results.append({
        "index": i,
        "reference": ref,
        "hypothesis": hyp,
        "wer": wer(ref_n, hyp_n),
        "cer": cer(ref_n, hyp_n),
    })

    if (i + 1) % 10 == 0:
        print(i + 1, round(np.mean([r["wer"] for r in results]) * 100, 2))

df = pd.DataFrame(results)
print(len(df))

In [ ]:
mean_wer = df["wer"].mean() * 100
mean_cer = df["cer"].mean() * 100

print(MODEL_ID)
print(len(df))
print(round(mean_wer, 2))
print(round(mean_cer, 2))
print("=" * 50)

best_5  = df.sort_values("wer").head(5)
worst_5 = df.sort_values("wer", ascending=False).head(5)

print("\nEn yaxsi 5:")
for i, row in enumerate(best_5.itertuples(), 1):
    print(i, round(row.wer * 100, 1), round(row.cer * 100, 1))
    print(row.reference)
    print(row.hypothesis)

print("\nEn pis 5:")
for i, row in enumerate(worst_5.itertuples(), 1):
    print(i, round(row.wer * 100, 1), round(row.cer * 100, 1))
    print(row.reference)
    print(row.hypothesis)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, color, m in [
    (axes[0], "wer", "steelblue", mean_wer),
    (axes[1], "cer", "coral", mean_cer),
]:
    ax.hist(df[col] * 100, bins=25, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(m, color="red", linestyle="--", label=round(m, 1))
    ax.set_title(col.upper() + " dagilimi")
    ax.set_xlabel(col.upper() + " (%)")
    ax.set_ylabel("sayi")
    ax.legend()

plt.suptitle(MODEL_ID + "  wer: " + str(round(mean_wer, 1)) + "  cer: " + str(round(mean_cer, 1)))
plt.tight_layout()
plt.savefig("asr_performance.png", dpi=150, bbox_inches="tight")
plt.show()

df[["index", "reference", "hypothesis", "wer", "cer"]].to_csv("asr_results.csv", index=False, encoding="utf-8")